# 00 — Environment Validation

**Purpose**: Verify that every required package is installed, GPU is accessible, raw data paths exist, and the `wildfire_gnn` source package is importable.

**Run this before any other notebook.**  
Every cell must pass without errors before you continue to Phase 1.

---

In [1]:
import os
import sys

# Add src/ to path so wildfire_gnn is importable without pip install -e .
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_PATH = os.path.join(PROJECT_ROOT, 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f'Project root : {PROJECT_ROOT}')
print(f'Python       : {sys.version}')

Project root : d:\wildfire\spatiotemporal_wildfire_gnn
Python       : 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:40:08) [MSC v.1938 64 bit (AMD64)]


In [2]:
# ── 1. Core scientific stack ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import scipy
import sklearn

print(f'numpy   {np.__version__}')
print(f'pandas  {pd.__version__}')
print(f'scipy   {scipy.__version__}')
print(f'sklearn {sklearn.__version__}')

numpy   1.26.4
pandas  2.2.1
scipy   1.12.0
sklearn 1.4.2


In [3]:
# ── 2. PyTorch + CUDA ─────────────────────────────────────────────────────
import torch

print(f'torch          {torch.__version__}')
print(f'CUDA available {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'CUDA device    {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU memory     {mem:.1f} GB')
else:
    print('WARNING: No GPU found — training will be slow on 300k nodes')

torch          2.1.2+cpu
CUDA available False


In [5]:
import torch
import torch_geometric
from torch_geometric.data import Data

print(f"torch_geometric {torch_geometric.__version__}")

_x = torch.randn(5, 3)
_ei = torch.tensor([[0, 1, 2], [1, 2, 3]], dtype=torch.long)

_g = Data(x=_x, edge_index=_ei)

assert _g.num_nodes == 5
assert _g.num_edges == 3

print("PyG Data smoke test PASSED")

torch_geometric 2.5.2
PyG Data smoke test PASSED


In [6]:
# ── 4. Geospatial stack ───────────────────────────────────────────────────
import rasterio
import geopandas as gpd
import pyproj
import shapely

print(f'rasterio   {rasterio.__version__}')
print(f'geopandas  {gpd.__version__}')
print(f'pyproj     {pyproj.__version__}')
print(f'shapely    {shapely.__version__}')

rasterio   1.3.9
geopandas  0.14.3
pyproj     3.6.1
shapely    2.0.3


In [7]:
# ── 5. Baseline models ────────────────────────────────────────────────────
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

print(f'xgboost {xgb.__version__}')
print(f'sklearn RF  importable')

xgboost 2.0.3
sklearn RF  importable


In [8]:
# ── 6. Visualization ─────────────────────────────────────────────────────
import matplotlib
import seaborn as sns

print(f'matplotlib  {matplotlib.__version__}')
print(f'seaborn     {sns.__version__}')

matplotlib  3.8.3
seaborn     0.13.2


In [12]:
import os
PROJECT_ROOT = os.getcwd()

In [13]:
# ── 7. Internal package ───────────────────────────────────────────────────
from wildfire_gnn.utils import load_yaml_config, set_seed, get_device, describe_device
from wildfire_gnn.utils import get_logger, section, success

print('wildfire_gnn package importable  PASSED')

# Load config
config = load_yaml_config(os.path.join(PROJECT_ROOT, 'configs', 'gnn_config.yaml'))
print(f'Config loaded, keys: {list(config.keys())}')

ImportError: cannot import name 'load_yaml_config' from 'wildfire_gnn.utils' (unknown location)

In [14]:
# ── 8. Seed + device ──────────────────────────────────────────────────────
set_seed(config['training']['seed'])
device = get_device()
print(f'Seed set   : {config["training"]["seed"]}')
print(f'Device     : {describe_device(device)}')

NameError: name 'set_seed' is not defined

In [15]:
# ── 9. Raw data paths ─────────────────────────────────────────────────────
from pathlib import Path

raw_files = [
    'Burn_Prob.img',
    'CFL.img',
    'FSP_Index.img',
    'Fuel_Models.img',
    'Ignition_Prob.img',
    'Struct_Exp_Index.img',
]

raw_dir = Path(PROJECT_ROOT) / config['paths']['raw_dir']
print(f'Raw data directory: {raw_dir}')
print(f'Exists: {raw_dir.exists()}')
print()

all_ok = True
for f in raw_files:
    fp = raw_dir / f
    status = '✓' if fp.exists() else '✗  MISSING'
    print(f'  {status}  {f}')
    if not fp.exists():
        all_ok = False

print()
if all_ok:
    print('All raw data files found  PASSED')
else:
    print('WARNING: Some raw files missing — update paths in configs/gnn_config.yaml')

NameError: name 'config' is not defined

In [ ]:
# ── 10. Directory structure ───────────────────────────────────────────────
required_dirs = [
    'data/raw',
    'data/interim/aligned',
    'data/processed',
    'data/features',
    'data/external',
    'reports/figures',
    'reports/tables',
    'notebooks',
    'scripts',
    'src/wildfire_gnn',
    'configs',
    'tests',
]

root = Path(PROJECT_ROOT)
for d in required_dirs:
    exists = (root / d).exists()
    status = '✓' if exists else '✗  CREATE'
    print(f'  {status}  {d}')

print()
print('Directory check complete.')


## Summary

If all cells above passed without red errors:

- Environment is correctly installed
- PyTorch Geometric is working
- Geospatial stack is functional
- Internal `wildfire_gnn` package is importable
- Config loads without errors
- Raw data paths are accessible

**You are ready to proceed to `01_dataset_exploration.ipynb`.**